In [2]:
from collections import defaultdict

corpus = [
    "best places to visit in india",
    "best places to visit in chennai",
    "best places to visit near me",
    "best places to visit during summer",
    "best restaurants near me",
    "best restaurants in chennai",
    "places to visit in kerala",
    "places to visit during winter"
]
D = 0.75

# Dictionaries for all n-grams
unigram = defaultdict(int)
bigram = defaultdict(int)
trigram = defaultdict(int)
fourgram = defaultdict(int)
fivegram = defaultdict(int)

history1 = defaultdict(int)
history2 = defaultdict(int)
history3 = defaultdict(int)
history4 = defaultdict(int)

for sentence in corpus:
    words = ["<s>"]*4 + sentence.lower().split() + ["</s>"]
    for w in words:
        unigram[w] += 1
    for i in range(len(words)-1):
        bg = tuple(words[i:i+2])
        bigram[bg] += 1
        history1[(words[i],)] += 1
    for i in range(len(words)-2):
        tg = tuple(words[i:i+3])
        trigram[tg] += 1
        history2[tuple(words[i:i+2])] += 1
    for i in range(len(words)-3):
        fg = tuple(words[i:i+4])
        fourgram[fg] += 1
        history3[tuple(words[i:i+3])] += 1
    for i in range(len(words)-4):
        fvg = tuple(words[i:i+5])
        fivegram[fvg] += 1
        history4[tuple(words[i:i+4])] += 1

# Followers
followers1 = defaultdict(set)
followers2 = defaultdict(set)
followers3 = defaultdict(set)
followers4 = defaultdict(set)

for g in bigram:
    followers1[g[:-1]].add(g[-1])
for g in trigram:
    followers2[g[:-1]].add(g[-1])
for g in fourgram:
    followers3[g[:-1]].add(g[-1])
for g in fivegram:
    followers4[g[:-1]].add(g[-1])

# Continuation counts
continuation = defaultdict(set)
for g in bigram:
    continuation[g[-1]].add(g[:-1])
total_unique_bigrams = len(bigram)

# Recursive Kneser-Ney
def kneser_ney(context, word):
    n = len(context)
    # Unigram probability
    if n == 0:
        return len(continuation[word]) / total_unique_bigrams
    if n == 1:
        table = bigram
        history = history1
        followers = followers1
    elif n == 2:
        table = trigram
        history = history2
        followers = followers2
    elif n == 3:
        table = fourgram
        history = history3
        followers = followers3
    else:
        table = fivegram
        history = history4
        followers = followers4
    gram = tuple(context + [word])
    count = table.get(gram, 0)
    history_count = history.get(tuple(context), 0)
    if history_count == 0:
        return kneser_ney(context[1:], word)
    first = max(count - D, 0) / history_count
    lamb = (D * len(followers[tuple(context)])) / history_count
    lower = kneser_ney(context[1:], word)
    return first + lamb * lower

# Prediction
query = "best places to visit"
context = query.lower().split()[-4:]
vocabulary = set(unigram.keys())
scores = {}
for word in vocabulary:
    if word in ["<s>", "</s>"]:
        continue
    scores[word] = kneser_ney(context, word)
prediction = sorted(scores.items(), key=lambda x:x[1], reverse=True)

print("Input :", query)
print("\nSuggestions")

for w,p in prediction[:5]:
    print(w, ":", round(p,4))

Input : best places to visit

Suggestions
in : 0.6347
during : 0.2413
near : 0.1005
places : 0.0025
me : 0.0012
